# Series & DataFrames — the two objects the rest of pandas is built from

01 Core Python · **▶ 02 Pandas** · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · 08 Case Studies · 09 Deployment

`02_Pandas_Essentials/01_series_and_dataframes.ipynb`

---

### In one paragraph (no jargon)

Pandas gives you exactly two containers, and every single thing you do from here on is one of them. A **Series** is one column of data with labels attached — think of a single column in Excel, plus the row numbers. A **DataFrame** is a bunch of Series stacked side by side sharing the same row labels — that's a whole spreadsheet. That's it. `groupby`, `merge`, `pivot_table` and everything else are just operations on these two. Get comfortable here and the rest of the course stops feeling like memorisation.

### After this notebook you can

- Create a Series and a DataFrame from lists, dictionaries and NumPy arrays
- Read the four numbers that describe any dataset: shape, dtypes, info, describe
- Select a column, several columns, and a single cell — and know which gives a Series vs a DataFrame
- Add, rename, reorder and delete columns
- Understand what the index is and why resetting it matters

**Assumed knowledge:** lists and dictionaries from `01_Python_Fundamentals/01`

### What's inside

1. What a Series actually is
2. What a DataFrame actually is
3. The first five commands you run on any new dataset
4. Selecting columns (and the Series/DataFrame trap)
5. Adding, renaming and removing columns
6. The index — the bit everyone ignores
7. ⚡ Method chaining with .assign() and .pipe()
8. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    return rebuild()


def rebuild_customer_data():
    """Recreate customer_data.csv (200 rows) with the same columns and behaviour."""
    rng = np.random.default_rng(42)          # fixed seed -> identical numbers every run
    n = 200
    frame = pd.DataFrame({
        'Customer_ID':     np.arange(1, n + 1),
        'Age':             rng.integers(18, 70, n).astype(float),
        'Gender':          rng.choice(['Male', 'Female'], n),
        'Income':          rng.normal(60000, 18000, n).round(-2).clip(20000, 150000),
        'Purchase_Amount': rng.gamma(4, 400, n).round(2),
        'Region':          rng.choice(['North', 'South', 'East', 'West'], n),
    })
    # the real file has a scattering of blanks — reproduce them so the cleaning code has work to do
    for col, frac in [('Age', .05), ('Income', .06), ('Purchase_Amount', .04)]:
        frame.loc[rng.choice(n, int(n * frac), replace=False), col] = np.nan
    return frame

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

Setup complete. pandas 3.0.2 | numpy 2.4.4


## 1. What a Series actually is

A Series is **values + an index**. The index is the set of labels down the left. If you don't supply one,
pandas numbers the rows 0, 1, 2… — but you can use anything as a label, and that's what makes a Series more
powerful than a plain list.

In [2]:
# Three ways to build a Series — all equivalent, choose by what you already have

# From a list: index is created automatically as 0, 1, 2, ...
monthly_sales = pd.Series([1200, 1500, 980, 2100])
print("From a list:\n", monthly_sales, "\n")

# With your own labels — now the row names carry meaning
monthly_sales = pd.Series([1200, 1500, 980, 2100],
                          index=['Jan', 'Feb', 'Mar', 'Apr'],
                          name='sales')          # naming the Series matters when it joins a DataFrame
print("With labels:\n", monthly_sales, "\n")

# From a dictionary: keys become the index automatically — usually the tidiest option
monthly_sales = pd.Series({'Jan': 1200, 'Feb': 1500, 'Mar': 980, 'Apr': 2100}, name='sales')
print("From a dict:\n", monthly_sales)

From a list:
 0    1200
1    1500
2     980
3    2100
dtype: int64 

With labels:
 Jan    1200
Feb    1500
Mar     980
Apr    2100
Name: sales, dtype: int64 

From a dict:
 Jan    1200
Feb    1500
Mar     980
Apr    2100
Name: sales, dtype: int64


In [3]:
# Why the index is worth having: you can look values up BY NAME
print("March sales      :", monthly_sales['Mar'])
print("Jan and Apr      :", monthly_sales[['Jan', 'Apr']].tolist())
print("By position (2nd) :", monthly_sales.iloc[1])

# And maths applies to the WHOLE Series at once — no loop anywhere
print("\nWith 18% GST added:\n", (monthly_sales * 1.18).round(2))

# Comparisons produce a Series of True/False — this is the seed of all filtering
above_target = monthly_sales > 1300
print("\nAbove target?\n", above_target)
print("\nOnly the months above target:\n", monthly_sales[above_target])

# The useful summary numbers
print(f"\ntotal={monthly_sales.sum():,}  mean={monthly_sales.mean():.0f}  "
      f"max={monthly_sales.max():,} (in {monthly_sales.idxmax()})")

March sales      : 980
Jan and Apr      : [1200, 2100]
By position (2nd) : 1500

With 18% GST added:
 Jan    1416.0
Feb    1770.0
Mar    1156.4
Apr    2478.0
Name: sales, dtype: float64

Above target?
 Jan    False
Feb     True
Mar    False
Apr     True
Name: sales, dtype: bool

Only the months above target:
 Feb    1500
Apr    2100
Name: sales, dtype: int64

total=5,780  mean=1445  max=2,100 (in Apr)


## 2. What a DataFrame actually is

A DataFrame is several Series sharing one index. Build one from a **dictionary of lists** — keys become column
names, lists become the columns. This is the single most common way you'll create test data in an exam when
you're not given a file.

In [4]:
# The exam-standard way to create a DataFrame: a dictionary of lists
sales_data = {
    'Branch':   ['Delhi', 'Mumbai', 'Delhi', 'Bengaluru', 'Mumbai', 'Bengaluru'],
    'Product':  ['Laptop', 'Laptop', 'Monitor', 'Laptop', 'Monitor', 'Mouse'],
    'Units':    [12, 18, 25, 9, 14, 60],
    'Revenue':  [660000, 990000, 375000, 495000, 210000, 30000],
}
df = pd.DataFrame(sales_data)
print("Every key becomes a column, every list becomes that column's values:\n")
df

Every key becomes a column, every list becomes that column's values:



,Branch,Product,Units,Revenue
0,Delhi,Laptop,12,660000
1,Mumbai,Laptop,18,990000
2,Delhi,Monitor,25,375000
3,Bengaluru,Laptop,9,495000
4,Mumbai,Monitor,14,210000
5,Bengaluru,Mouse,60,30000


In [5]:
# Other routes into a DataFrame — recognise them, you'll be handed all three at some point

# 1. A LIST OF DICTIONARIES — one dict per row. This is what an API or JSON file gives you.
records = [{'Branch': 'Delhi', 'Units': 12}, {'Branch': 'Mumbai', 'Units': 18}]
print("From a list of dicts:\n", pd.DataFrame(records), "\n")

# 2. A NUMPY ARRAY plus column names — common after a machine-learning step
array = np.array([[12, 660000], [18, 990000]])
print("From a NumPy array:\n", pd.DataFrame(array, columns=['Units', 'Revenue']), "\n")

# 3. A FILE — what you'll actually use most (see notebook 06 for the full tour)
customers = load_data('customer_data.csv', rebuild=rebuild_customer_data)
print("From a CSV:", customers.shape, "rows x columns")

From a list of dicts:
    Branch  Units
0   Delhi     12
1  Mumbai     18 

From a NumPy array:
    Units  Revenue
0     12   660000
1     18   990000 

Loaded 'customer_data.csv' from /home/claude/work/build/Python-BDA-Complete-Notes/datasets
From a CSV: (200, 6) rows x columns


## 3. The first five commands you run on any new dataset

Muscle memory. Whatever file you're handed tomorrow, run these five before you do anything else — they answer
"how big is it, what's in it, what's broken, and what does it look like".

| Command | Question it answers |
|---|---|
| `df.shape` | How many rows and columns? |
| `df.head()` | What does a row actually look like? |
| `df.info()` | What type is each column, and how many values are missing? |
| `df.describe()` | What's the range, average and spread of the numbers? |
| `df.isnull().sum()` | Exactly how many blanks per column? |

In [6]:
# The five-command opening move — do this on every dataset, every time
print("1. SHAPE  (rows, columns):", customers.shape, "\n")

print("2. HEAD — the first 5 rows:")
display(customers.head())

print("\n3. INFO — column types and non-null counts:")
customers.info()

1. SHAPE  (rows, columns): (200, 6) 

2. HEAD — the first 5 rows:


,Customer_ID,Age,Gender,Income,Purchase_Amount,Region
0,1,55.0,Male,70000.0,2400.0,West
1,2,45.0,Male,65000.0,1900.0,North
2,3,31.0,Male,30000.0,2400.0,East
3,4,59.0,Female,50000.0,1400.0,NaN
4,5,24.0,Female,70000.0,4000.0,East



3. INFO — column types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Customer_ID      200 non-null    int64  
 1   Age              194 non-null    float64
 2   Gender           175 non-null    str    
 3   Income           183 non-null    float64
 4   Purchase_Amount  193 non-null    float64
 5   Region           193 non-null    str    
dtypes: float64(3), int64(1), str(2)
memory usage: 9.5 KB


In [7]:
print("4. DESCRIBE — the numeric summary")
print("   Read it column by column: count tells you how many values are PRESENT")
print("   (a count below the row total means missing data), and comparing max to")
print("   75% is your instant outlier check.\n")
display(customers.describe().round(2))

print("\n5. ISNULL().SUM() — missing values, precisely")
missing = customers.isnull().sum()
print(missing[missing > 0] if missing.sum() else "No missing values.")
print(f"\nTotal cells missing: {customers.isnull().sum().sum()} "
      f"({customers.isnull().sum().sum() / customers.size:.1%} of all cells)")

# describe() ignores text columns by default. Ask for them explicitly:
print("\nText columns (include='object'):")
display(customers.describe(include='object'))

4. DESCRIBE — the numeric summary
   Read it column by column: count tells you how many values are PRESENT
   (a count below the row total means missing data), and comparing max to
   75% is your instant outlier check.



,Customer_ID,Age,Income,Purchase_Amount
count,200.00,194.0,183.00,193.00
mean,100.50,38.6,50273.22,2559.07
std,57.88,12.4,18101.72,1215.15
min,1.00,18.0,20000.00,500.00
25%,50.75,28.0,35000.00,1500.00
50%,100.50,40.0,55000.00,2500.00
75%,150.25,49.0,67500.00,3500.00
max,200.00,59.0,75000.00,4900.00



5. ISNULL().SUM() — missing values, precisely
Age                 6
Gender             25
Income             17
Purchase_Amount     7
Region              7
dtype: int64

Total cells missing: 62 (5.2% of all cells)

Text columns (include='object'):


,Gender,Region
count,175,193
unique,2,4
top,Male,West
freq,94,59


## 4. Selecting columns — and the trap

This is the single most common source of confusion, so learn the rule now:

| You write | You get back | Why |
|---|---|---|
| `df['Age']` | a **Series** (one column) | single square brackets, one name |
| `df[['Age']]` | a **DataFrame** (one column wide) | the inner `[ ]` is a *list* of names |
| `df[['Age', 'Income']]` | a **DataFrame** | a list of two names |

It matters because some methods only exist on one of them. If you get
`AttributeError: 'DataFrame' object has no attribute 'str'`, you almost certainly used double brackets where
you needed single ones.

In [8]:
# Single brackets -> Series.  Double brackets -> DataFrame.
one_column   = customers['Age']
still_a_frame = customers[['Age']]
two_columns  = customers[['Age', 'Income']]

print("customers['Age']          ->", type(one_column).__name__)
print("customers[['Age']]        ->", type(still_a_frame).__name__)
print("customers[['Age','Income']] ->", type(two_columns).__name__)

print("\nWhy it matters — .mean() on a Series gives ONE number:")
print("  ", round(one_column.mean(), 2))
print("…but on a DataFrame it gives one number PER COLUMN:")
print(two_columns.mean().round(2).to_string())

# Selecting a single cell: row label first, then column
print("\nOne cell by label   df.loc[0, 'Age']  ->", customers.loc[0, 'Age'])
print("One cell by position df.iloc[0, 1]     ->", customers.iloc[0, 1])

# Selecting columns by TYPE — very handy when you don't know the file
print("\nNumeric columns only:", list(customers.select_dtypes(include='number').columns))
print("Text columns only   :", list(customers.select_dtypes(include='object').columns))

customers['Age']          -> Series
customers[['Age']]        -> DataFrame
customers[['Age','Income']] -> DataFrame

Why it matters — .mean() on a Series gives ONE number:
   38.6
…but on a DataFrame it gives one number PER COLUMN:
Age          38.60
Income    50273.22

One cell by label   df.loc[0, 'Age']  -> 55.0
One cell by position df.iloc[0, 1]     -> 55.0

Numeric columns only: ['Customer_ID', 'Age', 'Income', 'Purchase_Amount']
Text columns only   : ['Gender', 'Region']


## 5. Adding, renaming and removing columns

Adding a column is just assignment to a name that doesn't exist yet. Because pandas works on whole columns at
once, `df['GST'] = df['Purchase_Amount'] * 0.18` computes all 200 rows in one line — no loop.

In [9]:
work = customers.copy()          # WHY copy? so the original stays intact for later cells

# ADD — assign to a name that doesn't exist yet. The maths runs on every row at once.
work['GST'] = work['Purchase_Amount'] * 0.18
work['Total_Payable'] = work['Purchase_Amount'] + work['GST']

# ADD a text/derived column with a condition (np.where = if/else for whole columns)
work['Segment'] = np.where(work['Income'] > 70000, 'Premium', 'Standard')

# INSERT at a specific position instead of at the end
work.insert(1, 'Row_Number', range(1, len(work) + 1))     # (position, name, values)

print(work[['Row_Number', 'Customer_ID', 'Purchase_Amount', 'GST', 'Total_Payable', 'Segment']].head())

   Row_Number  Customer_ID  Purchase_Amount    GST  Total_Payable   Segment
0           1            1           2400.0  432.0         2832.0  Standard
1           2            2           1900.0  342.0         2242.0  Standard
2           3            3           2400.0  432.0         2832.0  Standard
3           4            4           1400.0  252.0         1652.0  Standard
4           5            5           4000.0  720.0         4720.0  Standard


In [10]:
# RENAME — pass a dictionary {old: new}. Only the ones you list change.
work = work.rename(columns={'Purchase_Amount': 'Spend', 'Total_Payable': 'Spend_Incl_GST'})
print("After renaming:", list(work.columns), "\n")

# Bulk tidy-up: lowercase every column name and replace spaces with underscores.
# WHY: 'Purchase Amount' forces df['Purchase Amount']; purchase_amount allows df.purchase_amount
tidy = work.rename(columns=lambda c: c.strip().lower().replace(' ', '_'))
print("Tidied names:", list(tidy.columns), "\n")

# REMOVE — three ways, and the difference matters
dropped = work.drop(columns=['GST'])                  # returns a NEW frame; original untouched
print("drop(columns=...) leaves the original alone. Still there?", 'GST' in work.columns)

del work['Row_Number']                                # deletes immediately, in place
print("del removes immediately. Row_Number gone?", 'Row_Number' not in work.columns)

popped = work.pop('Segment')                          # removes AND hands you the column back
print("pop returns the column so you can reuse it:", popped.head(3).tolist())

# 🔧 CHANGE THIS: axis=1 means columns, axis=0 means rows.
#    df.drop(columns=['a'])  ==  df.drop('a', axis=1)     <- prefer the first, it can't be misread
print("\nDrop rows 0 and 1 instead:", work.drop(index=[0, 1]).shape, "vs original", work.shape)

After renaming: ['Customer_ID', 'Row_Number', 'Age', 'Gender', 'Income', 'Spend', 'Region', 'GST', 'Spend_Incl_GST', 'Segment'] 

Tidied names: ['customer_id', 'row_number', 'age', 'gender', 'income', 'spend', 'region', 'gst', 'spend_incl_gst', 'segment'] 

drop(columns=...) leaves the original alone. Still there? True
del removes immediately. Row_Number gone? True
pop returns the column so you can reuse it: ['Standard', 'Standard', 'Standard']

Drop rows 0 and 1 instead: (198, 8) vs original (200, 8)


## 6. The index — the bit everyone ignores until it bites

The index is the row-label column on the left. Two situations where it suddenly matters:

- **After filtering**, the index keeps the *original* numbers (0, 5, 9, …) — there are gaps. Use
  `.reset_index(drop=True)` to renumber from 0. Forget the `drop=True` and pandas keeps the old index as a
  new column, which is almost never what you want.
- **`.loc` uses labels; `.iloc` uses positions.** After a filter these stop agreeing, and that is the source
  of a large fraction of all pandas bugs.

In [11]:
high_spenders = customers[customers['Purchase_Amount'] > 2000]
print("After filtering, the index has GAPS — these are the ORIGINAL row numbers:")
print(high_spenders.index[:8].tolist(), "\n")

print("So .iloc[0] (first row by POSITION) and .loc[<first label>] agree...")
print("   .iloc[0] Customer_ID:", high_spenders.iloc[0]['Customer_ID'])
print("   .loc[", high_spenders.index[0], "] Customer_ID:", high_spenders.loc[high_spenders.index[0], 'Customer_ID'])
print("…but .loc[0] would raise a KeyError if row 0 didn't survive the filter.\n")

renumbered = high_spenders.reset_index(drop=True)     # drop=True discards the old index entirely
print("After reset_index(drop=True):", renumbered.index[:8].tolist())

# Setting a meaningful index — makes lookups by ID instant and readable
by_id = customers.set_index('Customer_ID')
print("\nLook a customer up directly by their ID:")
print(by_id.loc[7])

print("\n…and back again:", by_id.reset_index().columns.tolist()[:3])

After filtering, the index has GAPS — these are the ORIGINAL row numbers:
[0, 2, 4, 5, 6, 7, 8, 9] 

So .iloc[0] (first row by POSITION) and .loc[<first label>] agree...
   .iloc[0] Customer_ID: 1
   .loc[ 0 ] Customer_ID: 1
…but .loc[0] would raise a KeyError if row 0 didn't survive the filter.

After reset_index(drop=True): [0, 1, 2, 3, 4, 5, 6, 7]

Look a customer up directly by their ID:
Age                   55.0
Gender                Male
Income             25000.0
Purchase_Amount     3800.0
Region               South
Name: 7, dtype: object

…and back again: ['Customer_ID', 'Age', 'Gender']


### ⚡ Beyond the syllabus — method chaining with `.assign()` and `.pipe()`

Most people write five lines that each reassign `df`, which makes it impossible to see the pipeline and easy to run a cell twice and corrupt the data. **Chaining** expresses the whole transformation as one readable pipeline that never mutates the original — it's how professional pandas is written, and it reads top-to-bottom like a recipe.

In [12]:
# ---- THE COMMON WAY: five separate steps, original destroyed, order easy to break ----
step = customers.copy()
step = step.dropna(subset=['Income', 'Purchase_Amount'])
step['GST'] = step['Purchase_Amount'] * 0.18
step['Total'] = step['Purchase_Amount'] + step['GST']
step['Segment'] = np.where(step['Income'] > 70000, 'Premium', 'Standard')
step = step.sort_values('Total', ascending=False)

# ---- THE CHAINED WAY: one expression, original untouched, reads like a recipe ----
# .assign() adds columns; each `lambda d:` refers to the frame AS IT IS AT THAT POINT,
# so a column created on one line can be used on the next.
chained = (
    customers
    .dropna(subset=['Income', 'Purchase_Amount'])
    .assign(GST     = lambda d: d['Purchase_Amount'] * 0.18,
            Total   = lambda d: d['Purchase_Amount'] + d['GST'],      # GST already exists here
            Segment = lambda d: np.where(d['Income'] > 70000, 'Premium', 'Standard'))
    .sort_values('Total', ascending=False)
)

print("Identical result:", step.reset_index(drop=True).equals(chained.reset_index(drop=True)))
print("Original still untouched:", 'GST' not in customers.columns)
display(chained.head(3))

Identical result: True
Original still untouched: True


,Customer_ID,Age,Gender,Income,Purchase_Amount,Region,GST,Total,Segment
47,48,23.0,Female,65000.0,4900.0,South,882.0,5782.0,Standard
187,188,44.0,Female,70000.0,4900.0,North,882.0,5782.0,Standard
93,94,51.0,Male,55000.0,4900.0,North,882.0,5782.0,Standard


In [13]:
# .pipe() slots YOUR OWN function into a chain without breaking the flow
def add_value_tier(frame, cuts=(500, 1500)):
    """Label each customer by spend band. 🔧 CHANGE THIS: pass your own `cuts`."""
    low, high = cuts
    return frame.assign(
        Tier=pd.cut(frame['Purchase_Amount'],
                    bins=[-np.inf, low, high, np.inf],
                    labels=['Low', 'Mid', 'High'])
    )

result = (customers
          .dropna(subset=['Purchase_Amount'])
          .pipe(add_value_tier, cuts=(800, 2000))          # your function, mid-chain
          .groupby('Tier', observed=True)['Purchase_Amount']
          .agg(['count', 'mean', 'sum'])
          .round(2))

print("Spend tiers built and summarised in a single expression:\n")
display(result)
print("\nWhy this earns marks: no intermediate variables, nothing mutated, and the whole")
print("transformation can be read top-to-bottom as a sequence of named steps.")

Spend tiers built and summarised in a single expression:



,count,mean,sum
Tier,,,
Low,21,690.48,14500.0
Mid,54,1525.93,82400.0
High,118,3364.41,397000.0



Why this earns marks: no intermediate variables, nothing mutated, and the whole
transformation can be read top-to-bottom as a sequence of named steps.


---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Build from a dict of lists | `pd.DataFrame({'a': [1,2], 'b': [3,4]})` |
| Rows & columns count | `df.shape` |
| First / last rows | `df.head(10)` · `df.tail()` |
| Types + missing counts | `df.info()` |
| Numeric summary | `df.describe()` |
| Text summary | `df.describe(include='object')` |
| Missing per column | `df.isnull().sum()` |
| One column (Series) | `df['Age']` |
| Several columns | `df[['Age', 'Income']]` |
| Numeric columns only | `df.select_dtypes(include='number')` |
| Add a column | `df['GST'] = df['Amt'] * 0.18` |
| Add conditionally | `np.where(cond, 'A', 'B')` |
| Rename | `df.rename(columns={'old': 'new'})` |
| Drop a column | `df.drop(columns=['GST'])` |
| Renumber rows | `df.reset_index(drop=True)` |
| Set a meaningful index | `df.set_index('Customer_ID')` |
| Chain several steps | `df.assign(x=…).pipe(fn).sort_values(…)` |

### Adapting this in the exam

- Given a file instead of a dictionary? Swap `pd.DataFrame(data)` for `pd.read_csv('file.csv')`; every line after that is unchanged.
- Asked for a specific number of rows? `head(n)` takes an argument: `df.head(20)`.
- Asked to 'create a new column based on a condition'? That's `np.where` for two outcomes, `np.select` or `pd.cut` for more.

### Traps that cost marks

- `df['Age']` is a Series, `df[['Age']]` is a DataFrame. Half of all pandas errors trace back to this.
- `df.drop('col')` alone tries to drop a **row** named 'col'. Always write `df.drop(columns=['col'])`.
- Most pandas methods return a **new** object. `df.rename(...)` on its own changes nothing — you must assign the result back.
- After filtering, the index has gaps. `df.iloc[0]` still works; `df.loc[0]` may raise `KeyError`.
- `reset_index()` without `drop=True` keeps the old index as a new column called `index`.
- `describe()` silently skips text columns. If a column is missing from the output, check `df.dtypes` — it's probably stored as text.